[Workshop-Übersicht](../README.md) · [Technische Vorbereitung](../README_technical_preparation.md) · [Aufgaben](../tasks/README.md)

# Setup prüfen: Vier Speicher, ein gemeinsamer Einstieg

Wähle den Kernel **Python (rothstein-storage-workshop-2026)** und führe alle Zellen aus. Die technische Vorbereitung bietet drei Wege: **A: lokal mit Docker**, **B: lokal ohne Docker** mit MongoDB Community Server und Neo4j Desktop sowie **C: Codespaces**. Starte lokal beide Dienste gemäss Deinem gewählten Weg. Neo4j-Container und Desktop-Instanz dürfen nicht gleichzeitig dieselben lokalen Ports belegen; [Neo4j-Startanleitung](../docs/setup/NEO4J_START.md). In Codespaces werden sie automatisch eingerichtet. Für den Betrieb ohne Docker bleibt `OFFLINE_ONLY = False`.

Dieses Notebook prüft Paketversionen, Eingaben und tatsächlich ausgeführte Schreib-/Leseoperationen. Es importiert noch keine Workshopdaten. Eigene Probeobjekte werden nach erfolgreicher Prüfung gezielt entfernt.

In [ ]:
from pathlib import Path
import sys

# Standard: alle vier Speicher prüfen.
# Nur für eine eingeschränkte Diagnose ohne Dienste auf True setzen.
OFFLINE_ONLY = False

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "scripts/verify_setup.py").is_file()
     and (p / "data/raw/source_manifest.json").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Repo-Hauptordner nicht gefunden. Den vollständig entpackten Repo-Ordner öffnen.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Prüfumfang:", "Basis ohne MongoDB/Neo4j" if OFFLINE_ONLY else "alle vier Speicher")

## Umgebung und Eingaben prüfen

Ein grüner Verbindungsstatus allein genügt nicht: Der Check schreibt eine eindeutig markierte Probe und liest sie über eine neu geöffnete Verbindung beziehungsweise Datei. Anschliessend entfernt er nur diese Probe. Bei einem Fehler zeigt die Ausgabe den betroffenen Bereich; die Fehlerbehebung steht in der technischen Vorbereitung.

In [ ]:
from scripts.verify_setup import run_checks

setup_result = run_checks(offline=OFFLINE_ONLY)
if setup_result["status"] != "passed":
    raise RuntimeError("Setup noch nicht vollständig bereit. Die gemeldeten Fehler zuerst beheben.")

## Welche Dienste spreche ich an?

Lokal zeigen die Verbindungen auf Deinen Rechner, unabhängig davon, ob die Dienste in Docker oder nativ laufen. Nach einer Änderung der Verbindungseinstellungen den Kernel neu starten. In Codespaces zeigen sie auf die beiden Begleitcontainer. Die folgende Übersicht gibt keine Passwörter aus.

In [ ]:
import pandas as pd
from scripts.storage_runtime import connection_summary

pd.DataFrame(connection_summary().items(), columns=["System", "Verbindungsziel"])

## Der gemeinsame Datenbestand

Die UAP-Aufgaben starten mit demselben Snapshot. Die PDF-Auswahl motiviert unsere Recherche; die Speicheraufgaben nutzen den gesamten Katalog. Eine Zeile ist ein Katalogeintrag und nicht automatisch eine einzelne Sichtung.

In [ ]:
import json

entries = [json.loads(line) for line in (ROOT / "data/input/catalog_entries.jsonl").read_text(encoding="utf-8").splitlines()]
metrics = json.loads((ROOT / "data/input/reference_metrics.json").read_text(encoding="utf-8"))
frame = pd.DataFrame(entries)
print(f"Katalogeinträge: {len(frame)}")
print(f"Für Jahreszählung geeignet: {int(frame['annual_eligible'].sum())}")
selected_keys = ["DOW-UAP-D079", "DOW-UAP-D080", "DOW-UAP-D077", "FBI-UAP-D024", "FBI-UAP-D026", "DOS-UAP-D001", "DOS-UAP-D002"]
frame.loc[frame["source_key"].isin(selected_keys), ["source_key", "title", "incident_date_raw", "release_date", "annual_eligible"]].sort_values("source_key")

## Was bleibt nach einem Neustart erhalten?

Ein neuer Python-Zugriff ist noch kein Neustart des Datenbankdiensts. Für diese zusätzliche Prüfung verwendest Du im Terminal `python scripts/persistence_check.py write`, stoppst/startest die Dienste beziehungsweise denselben Codespace und führst danach `python scripts/persistence_check.py read` aus. Mit `python scripts/persistence_check.py cleanup` entfernst Du die Testdaten anschliessend. Die vollständige Anleitung steht in der technischen Vorbereitung.

Eigene Datenbanken bleiben getrennt vom Setup-Test und vom jeweils anderen Fall. Die vier Aufgaben und die Microblogging-Demonstrationen werden in den folgenden Modellpaketen ergänzt.

Für den [nativen Weg](../docs/setup/OHNE_DOCKER.md#6-stoppen-fortsetzen-und-persistenz) den MongoDB-Dienst und die gleiche Neo4j-Desktop-Instanz neu starten. Innerhalb einer Persistenzprüfung nicht zwischen Docker und nativen Datenbanken wechseln.

In [ ]:
from IPython.display import Markdown, display

if OFFLINE_ONLY:
    display(Markdown("**Basisprüfung erfolgreich. MongoDB und Neo4j sind weiterhin ungeprüft.** Setze `OFFLINE_ONLY = False`, sobald die Dienste laufen, und führe alle Zellen erneut aus."))
else:
    display(Markdown("**SETUP OK – alle vier Speicher sind erreichbar und haben den Schreib-/Lesetest bestanden.**"))